In [1]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split

import tensorflow
from tensorflow import keras 
from tensorflow.keras import Sequential
from keras.layers import Dense

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer  

In [2]:
df = pd.read_csv('concrete_data.csv')
df.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [3]:
X = df.drop(['Strength'],axis  = 1)
y = df['Strength']

In [4]:
pipe = Pipeline(steps=[
    ('Simple Imputer',SimpleImputer())
])

In [33]:
X_tr,X_tst,y_tr,y_tst = train_test_split(
    X,y,test_size= 0.2,random_state = 42
)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_tst = scaler.transform(X_tst)

In [34]:
X_tr = pipe.fit_transform(X_tr)
X_tst = pipe.transform(X_tst)

In [35]:
X_tr= pd.DataFrame(X_tr,columns = X.columns)
X_tst = pd.DataFrame(X_tst,columns=X.columns)

In [143]:
model = Sequential()
model.add(Dense(64,activation ='leaky_relu',input_dim = 8))
model.add(Dense(32,activation= 'leaky_relu'))
model.add(Dense(16,activation = 'leaky_relu'))
model.add(Dense(1))

c:\Users\Krish\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [144]:
model.compile(optimizer = 'rmsprop',loss = 'huber',metrics = ['mae','mse'])


In [ ]:
from tensorflow.keras import callbacks
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

history  = model.fit(X_tr,y_tr,validation_split= 0.2,epochs=50,callbacks = early_stop)

Epoch 1/50


21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 3.9650 - mae: 4.4319 - mse: 40.6017 - val_loss: 4.4035 - val_mae: 4.8693 - val_mse: 43.5127
Epoch 2/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.9141 - mae: 4.3818 - mse: 40.1736 - val_loss: 4.5870 - val_mae: 5.0762 - val_mse: 46.1319
Epoch 3/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.9539 - mae: 4.4175 - mse: 40.4193 - val_loss: 4.3041 - val_mae: 4.7708 - val_mse: 42.5505
Epoch 4/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.8544 - mae: 4.3157 - mse: 39.7484 - val_loss: 4.4289 - val_mae: 4.9019 - val_mse: 43.4199
Epoch 5/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.9227 - mae: 4.3914 - mse: 39.7888 - val_loss: 4.3804 - val_mae: 4.8448 - val_mse: 43.8157
Epoch 6/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 3.9074 - mae: 4.3740 - mse: 39.7298 - val_loss: 4.2598 - val_mae: 4.7275 - val_mse: 41.5570
Epoch 7/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.8247 - mae: 4.2872 - mse: 39.6798 - val_loss: 4.2

In [148]:
print(y.min(), y.max(),y.mean())

2.33 82.6 35.817961165048544


In [149]:
model.evaluate(X_tr,y_tr)

 1/26 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - loss: 4.1546 - mae: 4.6075 - mse: 52.2710

26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.3767 - mae: 3.8332 - mse: 30.8064


[3.3767080307006836, 3.833181142807007, 30.80640983581543]

In [225]:
from tensorflow.keras import activations
def build_model(hp):
   model = Sequential()

   units = hp.Int('layers',8,128,step = 8)
   model.add(Dense(units = units, activation = 'relu',input_dim = 8))
   model.add(Dense(1,activation = 'relu'))

   model.compile(optimizer = hp.Choice('optimizer',['adam','sigmoid','tanh',]),loss='huber',metrics  = ['mae'])
   return model

In [239]:
import keras_tuner as kt
tuner = kt.RandomSearch(
    build_model,
    objective="val_mae",
    max_trials=10,
    directory="mydir",
    project_name="concrete",
    overwrite=True
)

c:\Users\Krish\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [240]:
best_model = tuner.search(X_tr,y_tr,epochs = 20,validation_data = (X_tst,y_tst))

Trial 10 Complete [00h 00m 01s]

Best val_mae So Far: 10.212884902954102
Total elapsed time: 00h 00m 35s


In [241]:
tuner.get_best_hyperparameters()[0]
models = tuner.get_best_models(num_models = 2)
best_model = models[0]

c:\Users\Krish\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\saving\saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [242]:
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,281 (5.00 KB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 0 (0.00 B)

In [243]:
best_model.fit(X_tr,y_tr,validation_data =(X_tst,y_tst),initial_epoch = 20,epochs = 100,batch_size =32)

Epoch 21/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 10.1348 - mae: 10.6266 - val_loss: 9.5795 - val_mae: 10.0742
Epoch 22/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.9523 - mae: 10.4428 - val_loss: 9.4124 - val_mae: 9.9056
Epoch 23/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9.7979 - mae: 10.2851 - val_loss: 9.2647 - val_mae: 9.7578
Epoch 24/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9.6443 - mae: 10.1312 - val_loss: 9.1044 - val_mae: 9.5980
Epoch 25/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9.4798 - mae: 9.9690 - val_loss: 8.9650 - val_mae: 9.4560
Epoch 26/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9.3263 - mae: 9.8115 - val_loss: 8.8206 - val_mae: 9.3124
Epoch 27/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9.1613 - mae: 9.6451 - val_loss: 8.6758 - val_mae: 9.1662
Epoch 28/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.0062 - mae: 9.4922 - val_loss: 8.4532 - val_mae: 8.9416
Epoch 29/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/st

In [244]:

best_model.evaluate(X_tst,y_tst)


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.6006 - mae: 5.0658 


[4.600554466247559, 5.065762042999268]